# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed (uncomment if needed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata and initialize Dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Published date:", getattr(metadata, 'datePublished', 'N/A'))
print("Cite as:", getattr(metadata, 'citeAs', 'N/A'))


## 2. Data Overview
Review available record sets, fields, and their IDs as defined in the Croissant schema.

In [ ]:
# List available record sets and their fields using their '@id'.
print('Record Sets in this dataset:')
for rs in dataset.record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")

print('\nDetailed field information for each record set:')
for rs in dataset.record_sets:
    print(f"\nRecord set: {rs.name} (@id: {rs.id})")
    for field in rs.fields:
        print(f"    Field: @id={field.id}, name={field.name}, dataType={getattr(field, 'dataType', None)}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field references use their `@id` properties from above.

In [ ]:
# For this dataset, the main tabular data is typically in the first (or only) record set.
# Find all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Available record_set @id's:", record_set_ids)

# We'll use the first record set for demonstration
main_record_set_id = record_set_ids[0]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

main_df = dataframes.get(main_record_set_id, pd.DataFrame())
print(f"Columns for record set {main_record_set_id}:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filtering, normalization, and grouping. Use numeric and grouping fields via their `@id` strings.

> **Note:** Please update the field `@id`s below as appropriate if you wish to explore different fields.

In [ ]:
# Select a numeric field for analysis. Use the '@id' as column name from above.
# For this dataset, commonly found numeric fields are: 'cr:field_coverage', 'cr:field_mw', etc.
# For demonstration, let's use 'cr:field_coverage' (e.g., protein coverage) if it exists.
numeric_field = None
possible_numeric_fields = ['cr:field_coverage', 'cr:field_mw', 'cr:field_pi', 'cr:field_abundance_1']
for f in possible_numeric_fields:
    if f in main_df.columns:
        numeric_field = f
        break

if numeric_field is not None:
    print(f"Using numeric field: {numeric_field}")

    # Filter for values > a threshold (example: 10)
    threshold = 10
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()

    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize the field
    norm_col = numeric_field + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by another field: e.g., 'cr:field_description' if available (textual)
    group_field_candidates = ['cr:field_description', 'cr:field_accession']
    group_field = None
    for f in group_field_candidates:
        if f in filtered_df.columns:
            group_field = f
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No common numeric field found in dataset columns. Please check available columns.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

> Example: Histogram of the chosen numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and numeric_field in main_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available for histogram.")

## 6. Conclusion
This notebook showcased how to load, explore, filter, and visualize protein mass spectrometry data from a FAIR<sup>2</sup>-certified dataset using the `mlcroissant` library with references strictly to entity `@id`s.

- Dataset fields, record sets, and columns can be systematically referenced by their `@id`.
- Simple EDA and visualization steps enable insights suitable for more advanced statistical and ML workflows.

**For further exploration:** Try changing the field `@id`s and applying additional transformations or visualizations using the dataset's Croissant schema!